# Ariel 2025 — EDA (Exploratory Data Analysis)
Khám phá dữ liệu cho báo cáo: phổ target, phân bố transit depth, light curve thô, PCA explained variance, star metadata, missing/outlier. Lưu plot vào `plots/`.

In [ ]:
# === Setup: clone running branch and make modules importable ===
import subprocess, sys
from pathlib import Path

# Detect environment
IS_KAGGLE = Path("/kaggle").exists()

if IS_KAGGLE:
    REPO_URL = "https://github.com/Jun1801/ML_IT3190E_Project.git"
    CLONE_DIR = Path("/kaggle/working/ML_IT3190E_Project")
    import subprocess
    if not CLONE_DIR.exists():
        subprocess.run(["git", "clone", "--branch", "running", "--single-branch", REPO_URL, str(CLONE_DIR)], check=True)
        print("Cloned 'running' →", CLONE_DIR)
    else:
        subprocess.run(["git", "-C", str(CLONE_DIR), "pull"], check=True)

    for _p in [str(CLONE_DIR / "src"), "/kaggle/input/ariel-ml-src/src"]:
        if Path(_p).exists():
            sys.path.insert(0, _p); print("Using src from:", _p); break

    DATA_ROOT = Path("/kaggle/input/ariel-data-challenge-2025")
    OUTPUT_DIR = Path("/kaggle/working")
    PLOTS_DIR = OUTPUT_DIR / "plots"
else:
    # Local development
    current_dir = Path.cwd()
    root_dir = current_dir.parent if current_dir.name == "notebooks" else current_dir
    sys.path.insert(0, str(root_dir / "src"))
    print("Using local src from:", root_dir / "src")
    
    DATA_ROOT = root_dir / "data"
    OUTPUT_DIR = root_dir / "outputs"
    PLOTS_DIR = root_dir / "report" / "figures"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_ROOT exists:", DATA_ROOT.exists())


## 1. Load target + star metadata

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure plot style for elegant visual styling
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "font.family": "sans-serif",
    "axes.edgecolor": "#CBD5E1",
    "axes.linewidth": 1.0,
    "grid.color": "#F1F5F9",
    "grid.linestyle": "-",
    "grid.linewidth": 0.8,
    "xtick.color": "#64748B",
    "ytick.color": "#64748B",
    "axes.labelcolor": "#1E293B",
    "axes.titlecolor": "#0F172A",
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "legend.fontsize": 9,
    "legend.frameon": True,
    "legend.framealpha": 0.9,
    "legend.edgecolor": "#E2E8F0",
    "figure.titlesize": 14,
    "figure.titleweight": "bold"
})

targets = pd.read_csv(DATA_ROOT / "train.csv")
wl_cols = [c for c in targets.columns if c != "planet_id"]
Y = targets[wl_cols].to_numpy(dtype=float)
print("targets:", Y.shape, "| planets:", targets.shape[0], "| wavelengths:", len(wl_cols))

star_path = DATA_ROOT / "train_star_info.csv"
star = pd.read_csv(star_path) if star_path.exists() else None
print("star_info:", None if star is None else star.shape)


## 2. Phổ target: mean ± std + vài mẫu

In [ ]:
mean_spec = Y.mean(axis=0); std_spec = Y.std(axis=0)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: Mean spectrum with shaded uncertainty
ax[0].plot(mean_spec, color="#1E3A8A", lw=2, label="Mean Spectrum")
ax[0].fill_between(np.arange(len(mean_spec)), mean_spec - std_spec, mean_spec + std_spec,
                   alpha=0.15, color="#3B82F6", label="$\pm1$ Standard Deviation")
ax[0].set_title("Mean Target Spectrum $\overline{Y}$", fontsize=13, pad=10)
ax[0].set_xlabel("Wavelength Index ($\lambda$)", fontsize=11)
ax[0].set_ylabel("Transit Depth $(R_p/R_s)^2$", fontsize=11)
ax[0].legend(loc="upper right", frameon=True)
ax[0].grid(True, linestyle="--", alpha=0.4)

# Right plot: Sample target spectra with a beautiful color gradient
rng = np.random.default_rng(0)
sample_indices = rng.choice(Y.shape[0], size=min(6, Y.shape[0]), replace=False)
colors = plt.cm.plasma(np.linspace(0.1, 0.8, len(sample_indices)))

for idx, color in zip(sample_indices, colors):
    ax[1].plot(Y[idx], lw=1.0, color=color, alpha=0.8, label=f"Planet {targets['planet_id'].iloc[idx]}")

ax[1].set_title("Random Sample Target Spectra", fontsize=13, pad=10)
ax[1].set_xlabel("Wavelength Index ($\lambda$)", fontsize=11)
ax[1].set_ylabel("Transit Depth $(R_p/R_s)^2$", fontsize=11)
ax[1].grid(True, linestyle="--", alpha=0.4)

sns.despine(fig)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "eda_target_spectra.png", dpi=200, bbox_inches="tight")
plt.show()


## 3. Phân bố transit depth

In [ ]:
depth_per_planet = Y.mean(axis=1)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: Mean transit depth distribution
sns.histplot(depth_per_planet, bins=40, color="#0891B2", kde=True, ax=ax[0], edgecolor="white", alpha=0.6)
mean_val = depth_per_planet.mean()
ax[0].axvline(mean_val, color="#D97706", linestyle="--", lw=1.5, label=f"Mean: {mean_val:.4e}")
ax[0].set_title("Mean Transit Depth Distribution", fontsize=13, pad=10)
ax[0].set_xlabel("Mean Transit Depth $(R_p/R_s)^2$", fontsize=11)
ax[0].set_ylabel("Count", fontsize=11)
ax[0].legend(loc="upper right")
ax[0].grid(True, linestyle="--", alpha=0.4)

# Right plot: Per-wavelength standard deviation distribution (spectral variation)
sns.histplot(std_spec, bins=40, color="#EA580C", kde=True, ax=ax[1], edgecolor="white", alpha=0.6)
median_std = np.median(std_spec)
ax[1].axvline(median_std, color="#4F46E5", linestyle="--", lw=1.5, label=f"Median: {median_std:.4e}")
ax[1].set_title("Per-Wavelength Spectral Standard Deviation", fontsize=13, pad=10)
ax[1].set_xlabel("Wavelength Std $\sigma_\lambda$", fontsize=11)
ax[1].set_ylabel("Count", fontsize=11)
ax[1].legend(loc="upper right")
ax[1].grid(True, linestyle="--", alpha=0.4)

sns.despine(fig)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "eda_depth_distribution.png", dpi=200, bbox_inches="tight")
plt.show()
print(f"depth mean={depth_per_planet.mean():.4e} | spectral variation (median per-wl std)={np.median(std_spec):.4e}")


## 4. PCA explained variance (biện minh cho `n_components`)

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=min(80, Y.shape[1], Y.shape[0])).fit(Y)
cum = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(8, 5))
components = np.arange(1, len(cum) + 1)

# Plot cumulative variance
plt.plot(components, cum, "o-", color="#4F46E5", lw=2, ms=4, label="Cumulative Explained Variance")
plt.fill_between(components, cum, alpha=0.1, color="#6366F1")

# Draw thresholds and find crossings
thresholds = [0.90, 0.95, 0.99]
colors = ["#10B981", "#F59E0B", "#EF4444"]

for thr, color in zip(thresholds, colors):
    k = int(np.searchsorted(cum, thr) + 1)
    print(f"  cần {k:3d} components để giữ {thr*100:.0f}% variance")
    
    # Draw horizontal threshold line
    plt.axhline(y=thr, ls="--", lw=1.0, color=color, alpha=0.8)
    
    # Draw vertical line from crossing point to x-axis
    plt.axvline(x=k, ymax=(thr - plt.ylim()[0]) / (plt.ylim()[1] - plt.ylim()[0]), ls=":", lw=1.0, color=color, alpha=0.8)
    
    # Draw a point at the crossing
    plt.plot(k, cum[k-1], "o", color=color, ms=6)
    
    # Add text annotation
    plt.text(k + 1.5, thr - 0.025, f"$K={k}$ components ({thr*100:.0f}%)", color=color, fontsize=9, fontweight="bold")

plt.xlabel("Number of Principal Components ($K$)", fontsize=11)
plt.ylabel("Cumulative Explained Variance Ratio", fontsize=11)
plt.title("Target PCA — Explained Variance Curve", fontsize=13, pad=12)
plt.xlim(0, len(cum) + 1)
plt.ylim(min(cum) - 0.05, 1.02)
plt.legend(loc="lower right")
plt.grid(True, linestyle="--", alpha=0.4)
sns.despine()
plt.tight_layout()
plt.savefig(PLOTS_DIR / "eda_pca_variance.png", dpi=200, bbox_inches="tight")
plt.show()


## 5. Light curve thô (tín hiệu vật lý)
Calibrate + detrend một planet rồi vẽ FGS white-light + vài kênh AIRS theo thời gian — thấy vết lõm transit.

In [ ]:
from config import PreprocessConfig, DatasetConfig
from data_io import ArielDataRepository
from pipeline import ArielPreprocessFeaturePipeline

repo = ArielDataRepository(DatasetConfig(data_root=DATA_ROOT))
planets = repo.list_planet_ids("train")
if planets:
    pid = planets[0]
    pipe = ArielPreprocessFeaturePipeline(PreprocessConfig(target_time_bins=128, apply_cds=True, detrend_degree=2, smooth_window=5))
    obs = repo.load_observation("train", pid)
    
    # Call internal method to get bounds for visual shading
    processed_curves, bounds, metrics = pipe._process_to_light_curves(
        airs_signal=obs.airs_signal,
        fgs_signal=obs.fgs_signal,
        airs_calibration=obs.airs_calibration,
        fgs_calibration=obs.fgs_calibration,
        airs_adc=repo.get_adc_params("AIRS-CH0", pid),
        fgs_adc=repo.get_adc_params("FGS1", pid)
    )
    
    airs = np.nan_to_num(processed_curves.airs)
    fgs = np.nan_to_num(processed_curves.fgs)
    
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    
    # Shading function for transit phases
    def shade_transit_phases(axis):
        axis.axvspan(0, bounds.start, alpha=0.08, color="#64748B", label="Out-of-Transit (OOT)")
        axis.axvspan(bounds.start, bounds.ingress_end, alpha=0.12, color="#F59E0B", label="Ingress")
        axis.axvspan(bounds.ingress_end, bounds.egress_start, alpha=0.12, color="#10B981", label="In-Transit Floor")
        axis.axvspan(bounds.egress_start, bounds.end, alpha=0.12, color="#EF4444", label="Egress")
        axis.axvspan(bounds.end, len(fgs)-1, alpha=0.08, color="#64748B")
        
    # Left plot: FGS1 white-light curve
    ax[0].plot(fgs, color="#0F172A", lw=2, label="Calibrated Flux")
    shade_transit_phases(ax[0])
    ax[0].set_title(f"FGS1 White-Light Curve (Planet {pid})", fontsize=13, pad=10)
    ax[0].set_xlabel("Time Bin", fontsize=11)
    ax[0].set_ylabel("Normalised Flux", fontsize=11)
    ax[0].legend(loc="lower left", frameon=True, fontsize=8)
    ax[0].grid(True, linestyle="--", alpha=0.4)
    
    # Right plot: Selected AIRS light curves
    selected_wavelengths = np.linspace(0, airs.shape[1]-1, 5, dtype=int)
    colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(selected_wavelengths)))
    
    for j, color in zip(selected_wavelengths, colors):
        ax[1].plot(airs[:, j], lw=1.2, color=color, alpha=0.8, label=f"Wavelength {j}")
        
    shade_transit_phases(ax[1])
    ax[1].set_title("AIRS Spectroscopic Light Curves", fontsize=13, pad=10)
    ax[1].set_xlabel("Time Bin", fontsize=11)
    ax[1].set_ylabel("Normalised Flux", fontsize=11)
    ax[1].legend(loc="lower left", frameon=True, fontsize=8, ncol=2)
    ax[1].grid(True, linestyle="--", alpha=0.4)
    
    sns.despine(fig)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "eda_light_curve.png", dpi=200, bbox_inches="tight")
    plt.show()
else:
    print("Không thấy raw data train/ — bỏ qua light curve.")


## 6. Star metadata + missing/outlier

In [ ]:
if star is not None:
    num = star.select_dtypes("number").drop(columns=[c for c in ["planet_id"] if c in star.columns], errors="ignore")
    print(num.describe().T[["mean", "std", "min", "max"]])
    
    cols = num.columns
    n_cols = len(cols)
    grid_w = 4
    grid_h = int(np.ceil(n_cols / grid_w))
    
    fig, axes = plt.subplots(grid_h, grid_w, figsize=(15, 3.2 * grid_h))
    axes = axes.flatten()
    
    for i, col in enumerate(cols):
        ax = axes[i]
        sns.histplot(num[col].dropna(), bins=25, color="#0F766E", kde=True, ax=ax, edgecolor="white", alpha=0.7)
        ax.set_title(col, fontsize=11, fontweight="bold", pad=8)
        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.grid(True, linestyle="--", alpha=0.4)
        
    # Hide unused subplots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
        
    sns.despine(fig)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "eda_star_metadata.png", dpi=200, bbox_inches="tight")
    plt.show()

print("\nNaN trong target:", int(np.isnan(Y).sum()))
# outlier planet theo độ sâu (z-score > 4)
z = np.abs((depth_per_planet - depth_per_planet.mean()) / (depth_per_planet.std() + 1e-12))
print("Outlier planet (|z|>4) theo mean depth:", int((z > 4).sum()))


## Tổng kết EDA
Plot đã lưu trong `plots/`: `eda_target_spectra.png`, `eda_depth_distribution.png`, `eda_pca_variance.png`, `eda_light_curve.png`, `eda_star_metadata.png`. Dùng cho phần Data/EDA của báo cáo.